# Heed: train on your own voice (advanced)

This notebook trains a wake word tuned to your own microphone and voice. You add
a handful of real recordings of yourself (upload or record in the browser), and
training mixes in hundreds of synthetic speakers for robustness. At the end you
test it on your own voice right here, then download the model.

If you just want a generic model with zero recording, use the simpler
`heed_train_colab.ipynb` instead.

**Switch the runtime to GPU** (Runtime, Change runtime type, T4 GPU). It works on
CPU too, just slower. Project: https://github.com/AndreiBulzan/heed-wakeword

## 1. Install

In [ ]:
# A GPU runtime (Runtime > Change runtime type > T4 GPU) makes TTS and training much faster.
!pip install -q "heed-wakeword[tts,export]"
!heed --version

## 2. Browser-recording helper

Defines `record_to_wav()`, used by the recording and testing cells below. It needs
microphone permission in the Colab output frame. If recording does not work in
your browser, use the upload cells instead, they do the same job.

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode
from pathlib import Path
import subprocess, uuid

_RECORD_JS = '''
async function recordAudio(seconds) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream);
  const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(res => { rec.onstop = res; });
  rec.start();
  await new Promise(r => setTimeout(r, seconds * 1000));
  rec.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const buf = await (new Blob(chunks)).arrayBuffer();
  const bytes = new Uint8Array(buf);
  let bin = '';
  for (let i = 0; i < bytes.length; i++) bin += String.fromCharCode(bytes[i]);
  return btoa(bin);
}
'''

def record_to_wav(out_path, seconds=2):
    display(Javascript(_RECORD_JS))
    b64 = output.eval_js(f'recordAudio({seconds})')
    tmp = f'/tmp/{uuid.uuid4().hex}.webm'
    Path(tmp).write_bytes(b64decode(b64))
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', str(out_path)],
                   check=True, capture_output=True)
    return out_path

print('record_to_wav() ready')

## 3. Pick your phrase and settings

In [ ]:
PHRASE = "hey jasper"  #@param {type:"string"}
N_TTS_POSITIVES = 250  #@param {type:"integer"}
EPOCHS = 35  #@param {type:"integer"}

import re
PROJECT = re.sub(r'[^a-z0-9]+', '_', PHRASE.lower()).strip('_') or 'wake'
print('phrase:', repr(PHRASE), '| project:', PROJECT)

## 4. Create the project and download the voice

In [ ]:
!heed init {PROJECT} --phrase "{PHRASE}"
!heed download-tts

## 5. Add your positives

Give it 8 or more clips of you saying the phrase, varied in tone, distance, and
room. Either upload existing clips (next cell) or record in the browser (the cell
after). You can do both.

In [ ]:
# Upload clips (wav, mp3, m4a all fine; they are converted to 16 kHz mono).
from google.colab import files
from pathlib import Path
import subprocess

proj = Path(PROJECT)
(proj / 'positive').mkdir(parents=True, exist_ok=True)
print(f'Upload clips of you saying "{PHRASE}":')
uploaded = files.upload()
for i, (fn, data) in enumerate(uploaded.items()):
    tmp = f'/tmp/up_{i}_{Path(fn).name}'
    Path(tmp).write_bytes(data)
    out = proj / 'positive' / f'me_{i:03d}.wav'
    subprocess.run(['ffmpeg', '-y', '-i', tmp, '-ar', '16000', '-ac', '1', str(out)], check=True, capture_output=True)
print('positives now:', len(list((proj / 'positive').glob('*.wav'))))

In [ ]:
# Or record in the browser. Run this cell; for each prompt, press Enter then speak.
proj = Path(PROJECT)
N_RECORDINGS = 8  #@param {type:"integer"}
for i in range(N_RECORDINGS):
    input(f'Press Enter, then say "{PHRASE}" ({i + 1}/{N_RECORDINGS}, about 2s)... ')
    record_to_wav(proj / 'positive' / f'rec_{i:03d}.wav', seconds=2)
    print(f'  saved {i + 1}/{N_RECORDINGS}')
print('positives now:', len(list((proj / 'positive').glob('*.wav'))))

## 6. Synthesize negatives

Hard negatives (phonetic neighbors and common phrases) so the model learns the
boundary. Training adds many more on top of these.

In [ ]:
from heed.tts import synthesize_phrase, phonetic_neighbor_distractors
from heed.audio import save_wav

proj = Path(PROJECT)
(proj / 'negative').mkdir(parents=True, exist_ok=True)
distractors = phonetic_neighbor_distractors(PHRASE, max_neighbors=12)
k = 0
for text in distractors:
    for clip in synthesize_phrase(text, 2, seed=100 + k):
        save_wav(proj / 'negative' / f'neg_{k:03d}.wav', clip)
        k += 1
print(f'seed negatives: {k}  (distractors: {distractors[:6]} ...)')

## 7. Train

Uses your real positives plus `N_TTS_POSITIVES` synthetic speakers and a pool of
hard negatives. With your own voice in the mix, training also EQ-matches the
synthetic audio toward your microphone. The printed held-out evaluation is the
honest cross-speaker signal.

In [ ]:
!heed train {PROJECT} --epochs {EPOCHS} --tts-pos {N_TTS_POSITIVES}

## 8. Test it on your own voice

Record a clip and score it against the trained model. Run the cell again to retry.
A score above the threshold printed by training means it triggered.

In [ ]:
proj = Path(PROJECT)
test_wav = proj / 'my_test.wav'
input(f'Press Enter, then say "{PHRASE}" to test... ')
record_to_wav(test_wav, seconds=2)
!heed test {PROJECT} "{test_wav}"

## 9. Export and download

In [ ]:
!heed export {PROJECT}
import shutil
from google.colab import files
archive = shutil.make_archive(f'{PROJECT}_model', 'zip', f'{PROJECT}/export')
files.download(archive)

## What you got

A sub-250 KB model plus its `wake.json` contract, tuned to your voice but robust
across speakers. Run it in the browser (`examples/inference_browser/`), on a phone
(`examples/inference_react_native/assets/`), or in Python. See the export README in
the zip and the project docs for the deployment snippets.